In [ ]:
# secp256k1 parameters — the numbers that secure Bitcoin

# Field prime: coordinates live in F_P
SECP_P = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F

# Group order: scalars (private keys) live in Z_N  
SECP_N = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141

# Generator point G: the "starting point" for all key generation
SECP_GX = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798
SECP_GY = 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8

# Curve coefficients
A_COEFF = 0
B_COEFF = 7

print("=== secp256k1 Parameters ===")
print(f"Curve: y² = x³ + {A_COEFF}x + {B_COEFF}")
print(f"P = 2²⁵⁶ - 2³² - 977")
print(f"  = {SECP_P}")
print(f"  ({SECP_P.bit_length()} bits)")
print(f"\nN = {SECP_N}")
print(f"  ({SECP_N.bit_length()} bits)")
print(f"\nNon-singular check: 4(0)³ + 27(7)² = {4*0**3 + 27*7**2} ≠ 0  ✓")
print(f"\nP mod 4 = {SECP_P % 4}  (enables efficient square roots)")

# Module 3: Point Operations

This is where the algebraic structure from Module 1 meets the curves from Module 2.
We define an "addition" operation on curve points that satisfies all the group axioms.

## 3.1 Geometric Intuition

```
Point Addition P + Q:          Point Doubling 2P:

     ·  Q                           ·
    / \                            /|\
   /   \                          / | \
  P     \  ← secant line        P  | tangent line
   \     \                        \ |
    \     T                        \T
     \   /                          |
      \ /                           |
       R = P + Q  (reflect T)       R = 2P  (reflect T)
```

1. Draw a line through P and Q (or the tangent at P for doubling)
2. The line hits the curve at a third point T
3. Reflect T across the x-axis to get R = P + Q

## 3.2 The Formulas

In [14]:
from typing import Optional

class Point:
    """A point on secp256k1 (or infinity)."""
    def __init__(self, x: Optional[int] = None, y: Optional[int] = None):
        self.x = x
        self.y = y
    
    def is_infinity(self) -> bool:
        return self.x is None or self.y is None
    
    def copy(self) -> 'Point':
        return Point(self.x, self.y)
    
    def __eq__(self, other):
        if self.is_infinity() and other.is_infinity():
            return True
        return self.x == other.x and self.y == other.y
    
    def __repr__(self):
        if self.is_infinity():
            return "O (point at infinity)"
        return f"({hex(self.x)[:16]}..., {hex(self.y)[:16]}...)"

G = Point(SECP_GX, SECP_GY)
INFINITY = Point()  # The identity element

print(f"Generator G:     {G}")
print(f"Identity O:      {INFINITY}")

Generator G:     (0x79be667ef9dcbb..., 0x483ada7726a3c4...)
Identity O:      O (point at infinity)


In [15]:
def mod_inverse(a: int, p: int) -> int:
    """a⁻¹ mod p via Fermat's little theorem: a^(p-2) mod p."""
    return pow(a, p - 2, p)

def mod_sqrt(a: int, p: int) -> int:
    """√a mod p for p ≡ 3 (mod 4): a^((p+1)/4) mod p."""
    return pow(a, (p + 1) // 4, p)

def point_add(p1: Point, p2: Point) -> Point:
    """
    Add two distinct points on secp256k1.
    
    Slope:  λ = (y₂ - y₁) / (x₂ - x₁) mod P
    Result: x₃ = λ² - x₁ - x₂
            y₃ = λ(x₁ - x₃) - y₁
    """
    if p1.is_infinity():
        return p2.copy()
    if p2.is_infinity():
        return p1.copy()
    
    if p1.x == p2.x:
        if (p1.y + p2.y) % SECP_P == 0:
            return Point()  # P + (-P) = O
        return point_double(p1)
    
    lam = ((p2.y - p1.y) * mod_inverse(p2.x - p1.x, SECP_P)) % SECP_P
    x3 = (lam * lam - p1.x - p2.x) % SECP_P
    y3 = (lam * (p1.x - x3) - p1.y) % SECP_P
    return Point(x3, y3)

def point_double(p: Point) -> Point:
    """
    Double a point on secp256k1.
    
    Slope:  λ = 3x² / 2y mod P  (tangent to curve at P)
    Result: x₃ = λ² - 2x
            y₃ = λ(x - x₃) - y
    """
    if p.is_infinity() or p.y == 0:
        return Point()
    
    lam = (3 * p.x * p.x * mod_inverse(2 * p.y, SECP_P)) % SECP_P
    x3 = (lam * lam - 2 * p.x) % SECP_P
    y3 = (lam * (p.x - x3) - p.y) % SECP_P
    return Point(x3, y3)

def point_negate(p: Point) -> Point:
    """Negate: -P = (x, -y mod P)."""
    if p.is_infinity():
        return Point()
    return Point(p.x, (SECP_P - p.y) % SECP_P)

# Verify G is on the curve
lhs = (G.y * G.y) % SECP_P
rhs = (G.x ** 3 + 7) % SECP_P
print(f"G on curve? y² mod P == x³+7 mod P: {lhs == rhs}  ✓")

# Verify group properties
G2 = point_double(G)
print(f"\n2G = {G2}")
print(f"G + O = G? {point_add(G, INFINITY) == G}  ✓  (identity)")
neg_G = point_negate(G)
print(f"G + (-G) = O? {point_add(G, neg_G).is_infinity()}  ✓  (inverse)")

G on curve? y² mod P == x³+7 mod P: True  ✓

2G = (0xc6047f9441ed7d..., 0x1ae168fea63dc3...)
G + O = G? True  ✓  (identity)
G + (-G) = O? True  ✓  (inverse)


## 3.3 Scalar Multiplication (Double-and-Add)

**The most important operation in ECC.** Computing $k \times G$ gives us
a public key from a private key.

Naive approach: add G to itself $k$ times → $O(k)$ operations.  
**Double-and-add**: use binary representation of $k$ → $O(\log k)$ operations.

```
Example: 13 × P  (13 = 1101 in binary)

Step  Binary  Action             Result
─────────────────────────────────────────
  0   1       result += addend    P
      ─       addend = 2×addend   2P
  1   0       (skip add)          P
      ─       addend = 2×addend   4P
  2   1       result += addend    P + 4P = 5P
      ─       addend = 2×addend   8P
  3   1       result += addend    5P + 8P = 13P
```

In [16]:
def scalar_mult(k: int, p: Point) -> Point:
    """Compute k × P using double-and-add. O(log k) operations."""
    if k == 0 or p.is_infinity():
        return Point()
    k = k % SECP_N
    if k == 0:
        return Point()
    
    result = Point()  # Start at O (identity)
    addend = p.copy()
    
    while k > 0:
        if k & 1:
            result = point_add(result, addend)
        addend = point_double(addend)
        k >>= 1
    
    return result

# Verify: 3G computed two ways
G3_algo = scalar_mult(3, G)
G3_manual = point_add(G, point_double(G))
print(f"3G (double-and-add): {G3_algo}")
print(f"3G (G + 2G):         {G3_manual}")
print(f"Match: {G3_algo == G3_manual}  ✓")

# The fundamental property: N × G = O (wraps around)
# (Don't actually compute this — it would take forever)
# But we can verify: (N-1)×G + G = O
print(f"\nFundamental: N × G = O (point at infinity)")
print(f"This means private key space is cyclic with order N.")
print(f"N ≈ 1.16 × 10⁷⁷ — more than atoms in the observable universe.")

3G (double-and-add): (0xf9308a019258c3..., 0x388f7b0f632de8...)
3G (G + 2G):         (0xf9308a019258c3..., 0x388f7b0f632de8...)
Match: True  ✓

Fundamental: N × G = O (point at infinity)
This means private key space is cyclic with order N.
N ≈ 1.16 × 10⁷⁷ — more than atoms in the observable universe.


## 3.4 Compressed Public Keys

Since $y^2 = x^3 + 7$ has at most two solutions for $y$ given any $x$,
we only need to store $x$ plus one bit indicating which $y$ (even or odd).

```
Uncompressed: 65 bytes  [04 || x (32 bytes) || y (32 bytes)]
Compressed:   33 bytes  [02/03 || x (32 bytes)]
                         02 = even y,  03 = odd y
```

In [17]:
def serialize_compressed(p: Point) -> bytes:
    """Point → 33-byte compressed public key."""
    prefix = 0x03 if (p.y & 1) else 0x02
    return bytes([prefix]) + p.x.to_bytes(32, 'big')

def parse_compressed(data: bytes) -> Point:
    """33-byte compressed public key → Point."""
    x = int.from_bytes(data[1:], 'big')
    y2 = (pow(x, 3, SECP_P) + 7) % SECP_P
    y = mod_sqrt(y2, SECP_P)
    if (y & 1) != (data[0] == 0x03):
        y = SECP_P - y
    return Point(x, y)

# Demo: generate a key pair
import secrets
private_key = secrets.randbelow(SECP_N - 1) + 1
public_key = scalar_mult(private_key, G)
compressed = serialize_compressed(public_key)

print(f"=== Key Pair Generation ===")
print(f"Private key (d):  {hex(private_key)[:20]}...")
print(f"Public key (P = d×G):")
print(f"  x: {hex(public_key.x)}")
print(f"  y: {hex(public_key.y)}")
print(f"Compressed: {compressed.hex()}")
print(f"  Prefix 0x{compressed[0]:02x} → y is {'odd' if compressed[0] == 0x03 else 'even'}")

# Round-trip verification
recovered = parse_compressed(compressed)
print(f"\nRound-trip: {public_key == recovered}  ✓")

=== Key Pair Generation ===
Private key (d):  0x565b3c7040cabb4fbf...
Public key (P = d×G):
  x: 0x6914603749c86f07134d385c6cfa19cc3925094b11d2df840ef72de8f379b0fa
  y: 0x5d1457fb3e76793152d17c0ce5f8c3009a75574d17dc5aa5344d457cebd841ef
Compressed: 036914603749c86f07134d385c6cfa19cc3925094b11d2df840ef72de8f379b0fa
  Prefix 0x03 → y is odd

Round-trip: True  ✓


---

# Module 4: From ElGamal to ECC

The essay draws a parallel between ElGamal and ECC. Both rely on the hardness
of the **discrete logarithm problem**, but in different mathematical settings.

## 4.1 The Discrete Logarithm Problem

| Setting | Easy Direction | Hard Direction |
|---------|---------------|----------------|
| **ElGamal** (integers mod p) | $\beta = \alpha^a \bmod p$ | Given $\beta, \alpha, p$, find $a$ |
| **ECC** (elliptic curve) | $Q = k \times P$ | Given $Q, P$, find $k$ |

Both are "one-way": trivial to compute forward, infeasible to reverse.

**But ECC wins on efficiency.** A 256-bit ECC key provides the same security as a
3072-bit RSA/ElGamal key.

| Security Level | RSA/ElGamal Key | ECC Key | Ratio |
|---------------|----------------|---------|-------|
| 80-bit | 1024 bits | 160 bits | 6.4× |
| 128-bit | 3072 bits | 256 bits | 12× |
| 256-bit | 15360 bits | 512 bits | 30× |